# 1. Linear programme with Pyomo

Lecture: *Week 3 - Linear Programming 2*, PDF page 6 (*Solution with Pyomo (Modelling Language)*).

**The problem.** The Week 02 lecture LP, now written in **Pyomo**, a modelling language: you write the
variables, objective and constraints much as you would on paper, and a solver finds the answer.

maximise $P = 5x + 7y$ subject to $3x + 2y \le 144$, $x \ge 4y$, $x, y \ge 0$.
In the code, $x$ is `x[0]` and $y$ is `x[1]`.

**How to run it.** Run the first code cell first (it installs Pyomo and the HiGHS solver; the first time takes about a minute), then *Run → Run All Cells*, or run the cells one by one while you read.

**Expected result.** $x = 41.14$, $y = 10.29$, $P = 277.71$: point B, the same answer as Week 02's
`linprog` and Excel Solver demos.

*Original script: `5 Source Files/Python/linprog_pyomo.py`. Runs in the browser with HiGHS instead of GLPK; the model and answers are unchanged.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

Load Pyomo's modelling tools (`ConcreteModel`, `Var`, `Objective`, `Constraint`, `SolverFactory`, …).

In [ ]:
from pyomo.environ import *

A *concrete model* is a container: everything we attach to `model` becomes part of the LP.

In [ ]:
# Create a concrete model
model = ConcreteModel()

Two decision variables, `x[0]` ($x$) and `x[1]` ($y$). The domain `NonNegativeReals` already gives $x, y \ge 0$, so no separate non-negativity constraints are needed.

In [ ]:
# Decision variables (non-negative)
model.x = Var([0, 1], domain=NonNegativeReals)

The objective: maximise $5x + 7y$. The coefficients are in the list `f`, and `sum(...)` builds $f_0 x_0 + f_1 x_1$.

In [ ]:
# Objective function (maximize 5x0 + 7x1)
f = [5, 7]
model.obj = Objective(expr=sum(f[i] * model.x[i] for i in [0, 1]), sense=maximize)

The two constraints, written as on the slide.

In [ ]:
# Constraints
model.c1 = Constraint(expr=3*model.x[0] + 2*model.x[1] <= 144)  # 3x + 2y <= 144
model.c2 = Constraint(expr=model.x[0] >= 4*model.x[1])          # x >= 4y

`pprint()` shows the model as Pyomo stores it, so you can check it before solving. Pyomo moves everything to one side: $x \ge 4y$ appears as $4y - x \le 0$.

In [ ]:
# Print model before solving
print("Model formulation:")
model.pprint()

Solve with **HiGHS**, an open-source solver that also runs in the browser. `tee=False` asks for the solver's own log not to be printed.

In [ ]:
# Solve with HiGHS
solver = SolverFactory('appsi_highs')
results = solver.solve(model, tee=False)

Check that the solver finished with an *optimal* solution, then `display()` the variable values, the objective and each constraint's left-hand side (*Body*).

In [ ]:
# Check results
if (results.solver.status == SolverStatus.ok) and \
   (results.solver.termination_condition == TerminationCondition.optimal):
    print("\nOptimal solution found:")
    model.display()
else:
    print("\nSolver did not find an optimal solution. Status:", results.solver.status)

## What do you notice?

Which constraints are binding (have no spare) at the optimum?

<details><summary>Answer</summary>

Both. `c1` has *Body* 144, equal to its upper bound 144. `c2` has *Body* $4y - x \approx 0$: it is printed as a
tiny number such as `-3.55e-14`, which is floating-point rounding, not a real gap. The optimum is the corner
where the two lines $3x + 2y = 144$ and $x = 4y$ cross (point B in Week 02).
</details>